# Inspect a PE config

Reads a YAML config and reproduces the consistency checks and diagnostic plots from `PE_response.py` (SNR, mismatch, log-likelihood at truth, FD/TD comparisons, spectrogram). Nothing is written to disk — all output stays in the notebook.

In [ ]:
import cupy as cp
import matplotlib.pyplot as plt
import numpy as np
from lisaconstants import ASTRONOMICAL_YEAR
from lisaorbits import OEMOrbits
from mojito import MojitoL1File
from scipy.signal.windows import tukey

from src.io import param_load
from src.likelihood import LogLikelihood, RecoveryConfig
from src.noise import build_inv_covariance
from src.priors import build_priors
from src.utils import inband_freqs, inner_prod_tdi, mismatch_tdi
from src.waveform import (
    ResponseConfig,
    WaveformConfig,
    build_response,
    param_names_for,
)

In [ ]:
CONFIG_PATH = "config/config_inj_1PA_rec_0PA_eps_e-6.yaml"

cfg = param_load(CONFIG_PATH)
print(f"Loaded config: {CONFIG_PATH}")

In [ ]:
def _waveform_cfg(block):
    lmax_raw = block.get("lmax", None)
    return WaveformConfig(
        model=block["model"],
        dt=float(block["dt"]),
        T=float(block["T"]),
        mode_selection_threshold=float(block.get("mode_selection_threshold", 0.0)),
        evolve_chi1=bool(block.get("evolve_chi1", True)),
        include_1PA_amps=bool(block.get("include_1PA_amps", True)),
        inspiral_kwargs=dict(block.get("inspiral_kwargs") or {}),
        amplitude_kwargs=dict(block.get("amplitude_kwargs") or {}),
        summation_kwargs=dict(block.get("summation_kwargs") or {}),
        lmax=int(lmax_raw) if lmax_raw is not None else None,
    )


def _response_cfg(block, orbit_file):
    return ResponseConfig(
        orbit_file=orbit_file,
        tdi_gen=block["tdi_gen"],
        tdi_chan=block["tdi_chan"],
        order=int(block["order"]),
        offset=float(block["offset"]),
        n_samples_delay=int(block["n_samples_delay"]),
        t_buffer=float(block["t_buffer"]),
        flip_hx=bool(block.get("flip_hx", True)),
        is_ecliptic_latitude=bool(block.get("is_ecliptic_latitude", False)),
    )


def _emri_vector(emri_block, model):
    z = float(emri_block.get("z", 0.0))
    vec = []
    for n in param_names_for(model):
        val = float(emri_block[n])
        if n in ("M", "mu"):
            val *= (1.0 + z)
        vec.append(val)
    return vec


inj_wcfg = _waveform_cfg(cfg["Injection"]["Waveform"])
rec_wcfg = _waveform_cfg(cfg["Recovery"]["Waveform"])
resp_cfg = _response_cfg(cfg["Response"], cfg["Data"]["orbit_file"])

if inj_wcfg.model != rec_wcfg.model:
    print("Note: injection and recovery models differ — expect biases.")

print(f"Injection : {inj_wcfg.model}  evolve_chi1={inj_wcfg.evolve_chi1}  1PA_amps={inj_wcfg.include_1PA_amps}")
print(f"Recovery  : {rec_wcfg.model}  evolve_chi1={rec_wcfg.evolve_chi1}  1PA_amps={rec_wcfg.include_1PA_amps}  lmax={rec_wcfg.lmax}")
print(f"Response  : {resp_cfg.tdi_chan}  {resp_cfg.tdi_gen}  order={resp_cfg.order}")

In [ ]:
with MojitoL1File(cfg["Data"]["mojito_l1_file"]) as l1:
    ts           = l1.tdis.time_sampling
    t0_l1        = float(ts.t0)
    mojito_dt    = float(ts.dt)
    central_freq = float(l1.laser_frequency)

DT         = inj_wcfg.dt
oem_orbits = OEMOrbits.from_included("esa-trailing")
t0_orbits  = float(oem_orbits.t_start) + 10.0
T_response = (
    inj_wcfg.T
    + (2 * resp_cfg.offset + 2 * resp_cfg.n_samples_delay * DT) / ASTRONOMICAL_YEAR
)
t0_l0  = t0_l1 - resp_cfg.n_samples_delay * mojito_dt
t_init = t0_l0 - resp_cfg.offset

print(f"Mojito L1 : t0={t0_l1:.3f} s   dt={mojito_dt:.3f} s")
print(f"t_init    = {t_init:.3f} s")
print(f"T_response= {T_response:.6f} yr")

In [ ]:
use_gpu = bool(cfg["Sampler"]["use_gpu"])

print(f"Building injection response ({inj_wcfg.model}) …")
inj_response = build_response(inj_wcfg, resp_cfg, t_init, t0_orbits, T_response, use_gpu=use_gpu)

inj_params      = _emri_vector(cfg["Injection"]["EMRI"], inj_wcfg.model)
rec_truth_params = _emri_vector(cfg["Injection"]["EMRI"], rec_wcfg.model)

xyz_data = inj_response(*inj_params)
N_t = xyz_data.shape[1]

windowing   = bool(cfg["Sampler"]["windowing"])
filter_freq = bool(cfg["Sampler"]["filter_freq"])
window      = cp.asarray(tukey(N_t, alpha=0.01)) if windowing else cp.ones(N_t)
freqs_inband, mask = inband_freqs(N_t, DT, filter_freq=filter_freq)
xyz_data_fft = cp.fft.rfft(xyz_data * window, axis=1)[:, mask]

xyz_data_np = cp.asnumpy(xyz_data)
del inj_response, xyz_data
cp.get_default_memory_pool().free_all_blocks()

print(f"  N_t = {N_t}   n_inband = {int(mask.sum())}")

In [ ]:
print("Building inverse covariance from Mojito noise …")
inv_cov, psd_diag = build_inv_covariance(
    cfg["Data"]["noise_file"], central_freq,
    cp.asnumpy(freqs_inband), DT, N_t,
    channels=resp_cfg.tdi_chan,
)
print(f"inv_cov shape: {inv_cov.shape}")

In [ ]:
print(f"Building recovery response ({rec_wcfg.model}) …")
rec_response = build_response(rec_wcfg, resp_cfg, t_init, t0_orbits, T_response, use_gpu=use_gpu)

## Consistency checks

In [ ]:
param_names = rec_wcfg.param_names()
x_I0_index  = param_names.index("x_I0") if "x_I0" in param_names else None
fixed_names = list(cfg["Sampler"].get("fixed_params", []) or [])

priors, bounds, sampled_idx = build_priors(
    param_names, rec_truth_params, fixed_names,
    n=float(cfg["Sampler"]["d"]), use_cupy=use_gpu,
)
fixed_idx = {
    param_names.index(n): rec_truth_params[param_names.index(n)]
    for n in fixed_names if n in param_names and n != "x_I0"
}

llike = LogLikelihood(
    data_fft=xyz_data_fft,
    inv_cov=inv_cov,
    recovery_response=rec_response,
    cfg=RecoveryConfig(
        param_names=param_names,
        fixed_params=fixed_idx,
        x_I0_index=x_I0_index,
    ),
    window=window,
    mask=mask,
)

xyz_rec_true_td  = rec_response(*rec_truth_params)
xyz_rec_true_fft = cp.fft.rfft(xyz_rec_true_td * window, axis=1)[:, mask]

snr      = float(cp.sqrt(inner_prod_tdi(xyz_data_fft, xyz_data_fft, inv_cov)))
mm       = mismatch_tdi(xyz_data_fft, xyz_rec_true_fft, inv_cov)
ll_truth = float(llike([rec_truth_params[i] for i in sampled_idx]))

print(f"SNR                                = {snr:.2f}")
print(f"Mismatch (injection vs recovery)   = {mm:.3e}")
print(f"loglike at truth                   = {ll_truth:.3e}")
if snr < 20:
    print("WARNING: Injection SNR below 20 — may not be recoverable.")

## Diagnostic plots

In [ ]:
freqs_np    = cp.asnumpy(freqs_inband)
data_fft_np = cp.asnumpy(xyz_data_fft)
rec_fft_np  = cp.asnumpy(xyz_rec_true_fft)
psd_diag_np = cp.asnumpy(psd_diag) if hasattr(psd_diag, 'get') else np.asarray(psd_diag)

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(freqs_np, 2 * freqs_np * np.abs(data_fft_np[0]),
          label="Injection", alpha=0.8)
ax.loglog(freqs_np, 2 * freqs_np * np.abs(rec_fft_np[0]),
          label="Recovery at truth", alpha=0.8, ls="--")
ax.loglog(freqs_np, np.sqrt(freqs_np * psd_diag_np[0]),
          label="Noise ASD (X)", ls=":", color="k")
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Characteristic strain")
ax.set_title(f"{cfg['Sampler']['name']} — X channel (FD)")
ax.grid(which="both", alpha=0.4)
ax.legend()
plt.show()

In [ ]:
xyz_rec_true_np = cp.asnumpy(xyz_rec_true_td)
t_axis = t_init + np.arange(N_t) * DT

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(t_axis, xyz_data_np[0],     lw=0.6, label="Injection")
axes[0].plot(t_axis, xyz_rec_true_np[0], lw=0.6, ls="--", label="Recovery at truth")
axes[0].set_ylabel("TDI X")
axes[0].legend()
axes[1].plot(t_axis, xyz_data_np[0] - xyz_rec_true_np[0], lw=0.6, color="C3")
axes[1].set_ylabel("Residual")
axes[1].set_xlabel("Time [s]")
for ax in axes:
    ax.grid(alpha=0.4)
fig.suptitle(f"{cfg['Sampler']['name']} — X channel (TD)")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
f_min_plot     = 1e-5
fs             = 1 / DT
required_nfft  = int(np.ceil(fs / f_min_plot))
N_fft          = min(xyz_data_np.shape[1], max(256, required_nfft))
ax.specgram(xyz_data_np[0], NFFT=N_fft, Fs=fs, cmap="viridis")
ax.set_yscale("log")
ax.set_ylim(f_min_plot, fs / 2)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Frequency [Hz]")
ax.set_title(f"{cfg['Sampler']['name']} — spectrogram of injected signal (X channel)")
plt.show()